In [ ]:
!git clone https://github.com/hazerMf/slm/
!cp -r slm/* /kaggle/working/

In [ ]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-3-1b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    token=HF_TOKEN
)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import os
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

food_dir = "/kaggle/working/food"
texts = []

for root, _, files in os.walk(food_dir):
    for file in files:
        if file.endswith(".txt"):
            file_path = os.path.join(root, file)
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read().strip()
                if content:
                    texts.append(content)

max_length = 512
all_chunks = []
for text in texts:
    tokens = tokenizer(text, truncation=False, return_attention_mask=False)["input_ids"]
    for i in range(0, len(tokens), max_length):
        chunk = tokens[i:i + max_length]
        if len(chunk) > 10:
            all_chunks.append(chunk)

dataset = Dataset.from_dict({"input_ids": all_chunks})
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
from transformers import Trainer, TrainingArguments

output_dir = "/kaggle/working/gemma_lora_output"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

trainer.train()

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/gemma_lora_output", "zip", output_dir)